## 调用其他前沿模型的 API（不只用 OpenAI）

本练习对应课程 **Day 2**：用 **OpenAI 兼容客户端**（`OpenAI(..., base_url=...)`）去打 **Gemini** 与本地 **Ollama**，对 YouTube 字幕做摘要。

核心思路：同一套 `chat.completions.create` 调用方式，换 `base_url` / `api_key` / `model` 就能切换后端——省钱、可本地跑。


## 练习目标：少花 OpenAI 的钱

用 **Google Gemini**（OpenAI 兼容端点）和本机 **Ollama** 做 YouTube 视频摘要，避免每次摘要都打付费 OpenAI API。

### 你会练到

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| OpenAI 兼容 API | `OpenAI(api_key=..., base_url=gemini_base)` |
| `messages`（system / user） | system 定「只根据字幕摘要」，user 塞字幕文本 |
| 环境变量 | `.env` 里的 `GOOGLE_API_KEY` |
| 本地 Ollama | `base_url=http://localhost:11434/v1` |

### 怎么跑

1. 安装依赖格 → 导入 → `load_dotenv` → 建 `gemini` 客户端
2. 定义 `system_prompt` 与 `extract_video_id` / `summarize_youtube_video`
3. 先用 Gemini 摘要一条视频，再 `ollama pull` 后换本地模型对比


In [86]:
# ========== 安装：YouTube 字幕库（quiet 模式）==========

# !pip：在 Jupyter 里执行 shell 安装；-q 减少安装日志噪音
# youtube-transcript-api：按视频 ID 拉取官方/社区字幕（Transcript），无需下载整段视频
!pip install -q youtube-transcript-api


In [87]:
# ========== 导入：环境、展示、OpenAI 兼容客户端、URL 解析、字幕 API ==========

# 导入标准库 os：后面用 getenv 读 GOOGLE_API_KEY 等环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮渲染摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：官方 SDK；也可指向 Gemini / Ollama 的兼容端点
from openai import OpenAI
# 从 urllib.parse 导入 urlparse / parse_qs：解析 YouTube 链接，抽出视频 ID
from urllib.parse import urlparse, parse_qs
# 从 youtube_transcript_api 导入 YouTubeTranscriptApi：按 video_id 拉取字幕条目
from youtube_transcript_api import YouTubeTranscriptApi


In [88]:
# ========== 环境变量 + Gemini 的 OpenAI 兼容 Base URL ==========

# override=True：.env 里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境读取 Google API Key（变量名必须是 GOOGLE_API_KEY，与 Gemini 端点配套）
api_key = os.getenv("GOOGLE_API_KEY")
# Gemini 的 OpenAI 兼容网关地址（注意末尾路径 /v1beta/openai/）；字符串影响实际请求目标，勿改译
gemini_base = "https://generativelanguage.googleapis.com/v1beta/openai/"


In [89]:
# ========== 创建指向 Gemini 的 OpenAI 兼容客户端 ==========

# api_key 用上格读到的 GOOGLE_API_KEY；base_url 指向 Gemini 兼容端点（不是官方 api.openai.com）
# 之后 gemini.chat.completions.create(...) 的用法与 OpenAI 相同
gemini = OpenAI(api_key=api_key, base_url=gemini_base)


In [136]:
# ========== system prompt：定「只根据字幕做摘要」的角色与输出格式 ==========

# system_prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
# 要点：只依据 transcript、不编造事实、输出短摘要 + 要点列表
system_prompt = """
You are a transcript-based YouTube summarizer.

Your job:
- Summarize ONLY from the transcript text provided by the user.
- Work for any topic (news, music, podcasts, tutorials, interviews, politics, etc.).

Rules:
1) Do not invent facts, names, dates, or speaker identities.
2) If something is unclear or likely mistranscribed, label it as [unclear].
3) If the transcript is noisy/incomplete, say what is uncertain.
4) Keep the summary concise and neutral.
5) Preserve important proper nouns exactly as they appear in transcript.

Output format:
- (1-2 sentences)
- Key points (4-8 bullets)
- People/organizations mentioned
- Claims that could not be verified from transcript
"""


#### 写一个函数：从 YouTube 链接抽出视频 ID，再拉取字幕（Transcript）

下面两格：先 `extract_video_id` 解析 URL，再 `summarize_youtube_video` 拉字幕并调用模型。


In [132]:
# ========== extract_video_id：从常见 YouTube URL 形态里取出视频 ID ==========

# 入参 url：完整观看链接；返回值是 11 位左右的 video_id，解析失败则 None
def extract_video_id(url: str) -> str:
    # urlparse：拆成 scheme / netloc / path / query 等部件
    p = urlparse(url)
    # 短链 youtu.be/<id>：ID 就在 path 里（去掉前导 /）
    if p.netloc in ("youtu.be", "www.youtu.be"):
        return p.path.lstrip("/")
    # 标准 watch 链接：query 里有 v=<id>
    if "youtube.com" in p.netloc:
        # parse_qs 把 query 变成 dict[str, list]；取不到 v 时用 [None] 兜底
        return parse_qs(p.query).get("v", [None])[0]
    # 其他域名：本函数不认识，返回 None 让上层报 Invalid YouTube URL
    return None


In [173]:
# ========== summarize_youtube_video：拉字幕 → 选模型 → Chat Completions ==========

# 导入字幕相关异常：字幕被关 / 找不到可用语言时会抛这些
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound

# ai：OpenAI 兼容客户端（gemini 或 ollama）；video_url：YouTube 链接；model_：Ollama 时指定别名
def summarize_youtube_video(ai, video_url, model_=None):
    # 1) 从 URL 抽出 video_id；无效则抛错（英文文案保持原样，供程序/用户识别）
    video_id = extract_video_id(video_url)
    if not video_id:
        raise ValueError("Invalid YouTube URL")
    # 2) 创建字幕 API 实例，再按视频 ID 拉取英文字幕（languages=['en']）
    ytt = YouTubeTranscriptApi()
    try:
        transcript = ytt.fetch(video_id, languages=['en'])
    except (TranscriptsDisabled, NoTranscriptFound):
        # 无字幕可用：向上抛出明确错误（字符串保持英文）
        raise ValueError("Transcript not available for this video")
    # 3) 把多条字幕片段的 .text 用空格拼成一整段纯文本
    transcript_text = " ".join(x.text for x in transcript)
    # 4) 按客户端身份选择具体 model 字符串（身份用 `is` 比较对象本身）
    if ai is gemini:
        # Gemini 侧模型 id：gemini-2.5-flash（字符串影响计费/能力，勿改译）
        m = "gemini-2.5-flash"
    elif ai is ollama:
        # Ollama：用 model_ 别名映射到本机已 pull 的模型名
        if model_ == "deepseek":
            m = "deepseek-r1:1.5b"
        elif model_ == "llama3.2":
            m = "llama3.2"
        elif not model_:
            # 走 Ollama 却没指定模型：明确报错提示（英文保持）
            raise ValueError("For Ollama, please specify a model (e.g. 'deepseek')")
    else:
        # 既不是 gemini 也不是 ollama：不支持
        raise ValueError("Unsupported model")
    # 5) 统一走 OpenAI 兼容的 chat.completions.create：system + user（字幕正文）
    response = ai.chat.completions.create(
        model = m,
        messages=[
            {"role": "system", "content": system_prompt},
            # user prompt 保留英文前缀；真正内容是 transcript_text
            {"role": "user", "content": f"Summarize this transcript:\n\n {transcript_text}"}
        ]
    )
    # 打印本次实际用的模型名，便于对比 Gemini / llama / deepseek
    print(f"Model used: {m}")
    # 返回助手生成的完整摘要字符串
    return response.choices[0].message.content


In [157]:
# ========== summarize_video：调用摘要函数，再用 Markdown 展示 ==========

# 薄包装：业务逻辑在 summarize_youtube_video；本函数负责笔记本里的展示体验
def summarize_video(ai, video_url, model_=None):
    # 拿到模型返回的 Markdown/纯文本摘要
    summary = summarize_youtube_video(ai, video_url, model_)
    # 先显示一个三级标题「Summary」（标题字符串保持英文，与原输出一致）
    display(Markdown(f"### Summary"))
    # 再把摘要正文渲染成 Markdown（列表、加粗等会更好看）
    display(Markdown(summary))



In [159]:
# ========== 试跑 A：用 Gemini 摘要一条 YouTube 视频 ==========

# 示例视频 URL（可改成你自己的链接；须有英文字幕，languages=['en']）
video_url = "https://www.youtube.com/watch?v=c4pQVTVlaKc"

# ai=gemini：走 Google 兼容端点；model_ 默认 None（函数内会固定选 gemini-2.5-flash）
summarize_video(gemini, video_url)


Model used: gemini-2.5-flash


### Summary

The speaker details a series of aggressive actions taken against drug cartels and the government of Venezuela, including designating cartels as foreign terrorist organizations and fentanyl as a weapon of mass destruction. A new military campaign is credited with stopping record amounts of drugs, eliminating a major cartel kingpin, and culminating in the defeat and capture of Venezuelan dictator Nicholas Maduro. The speaker also notes cooperation with Deli Rodriguez, described as the new president of Venezuela, to foster economic gains and hope for the country.

Key points:
*   Large parts of Mexico are controlled by murderous drug cartels, which have been designated as foreign terrorist organizations.
*   Illicit fentanyl has been declared a weapon of mass destruction.
*   A new military campaign has reportedly stopped record amounts of drugs, with sea routes virtually completely blocked.
*   One of the most sinister cartel kingpins has been taken down.
*   America's armed forces defeated an [unclear] enemy, ending the reign of Nicholas Maduro, who was brought to face American justice.
*   This is described as a colossal victory for US security and a new beginning for Venezuela.
*   The US is working with the "new president of Venezuela," Deli Rodriguez, for economic gains.
*   Maduro's heavily protected military fortress, guarded by thousands of soldiers and Russian and Chinese military technology, was swiftly descended upon.

People/organizations mentioned:
*   Nicholas Maduro
*   Deli Rodriguez (new president of Venezuela)
*   Drug cartels
*   America's armed forces
*   Russian military technology
*   Chinese military technology

Claims that could not be verified from transcript:
*   "stopped record amounts of drugs coming into our country and virtually stopped it completely coming in by water or sea."
*   "America's armed forces overwhelmed all defenses and utterly defeated a enemy. good fighters"
*   "This was an absolutely colossal victory for the security of the United States"
*   "This was a major military installation protected by thousands of soldiers and guarded by Russian and Chinese military technology."

### 接下来试试 Ollama！

用本机 Ollama 的 OpenAI 兼容接口（`/v1`）跑同一条 `summarize_video`，对比云端 Gemini 与本地模型的摘要风格与速度。


（作者说明：Ollama 上的 3.3 / 4 系列对本机太大。）

下面改用 **llama3.2**：体积更小，但仍够做字幕摘要实验。


In [46]:
# ========== 拉取本地模型：llama3.2 ==========

# ollama pull：从 Ollama 库下载模型到本机；名字必须与后面 model 字符串一致
!ollama pull llama3.2


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 2.3 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   0% ▕                  ▏ 7.5 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  11 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  14 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  20 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  21 MB/2.0 GB                  pulling manifest 
pulling dde5aa3fc5ff:   1% ▕                  ▏  26 MB/2.0 GB                

In [47]:
# ========== 确认 Ollama CLI 版本 ==========

# 排查用：看本机 ollama 是否安装成功、版本是否过旧
!ollama --version


ollama version is 0.17.0


In [48]:
# ========== 列出本机已安装的模型 ==========

# ollama list：确认 llama3.2（以及后面的 deepseek）是否已 pull 完成
!ollama list


NAME               ID              SIZE      MODIFIED       
llama3.2:latest    a80c4f17acd5    2.0 GB    57 seconds ago    
phi3:latest        4f2222927938    2.2 GB    4 days ago        
gemma3:270m        e7d36fb2c3b3    291 MB    4 days ago        


In [51]:
# ========== 探测本机 Ollama HTTP 服务是否在跑 ==========

# 注释说明：检查 Ollama 是否真的在监听默认端口
#To check if Ollama is effectivelly running:
# 导入 requests：用 HTTP GET 打本地健康检查端点
import requests
# 默认端口 11434；成功时 .content 多为欢迎字节串；失败则连接错误
requests.get("http://localhost:11434").content


b'Ollama is running'

In [52]:
# ========== 创建指向本地 Ollama 的 OpenAI 兼容客户端 ==========

# base_url 必须带 /v1（OpenAI 兼容路径）；api_key 对本地服务多为占位字符串 "ollama"
# 变量名 ollama 很关键：上面 summarize_youtube_video 用 `ai is ollama` 判断后端
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [163]:
# ========== 试跑 B：同一视频，改用本地 llama3.2 ==========

# 可复用上格的 video_url，或在此重新赋值（保持与 Gemini 试跑同一条便于对比）
video_url = "https://www.youtube.com/watch?v=c4pQVTVlaKc"

# 第三个参数 "llama3.2"：映射到本机模型名 llama3.2（见 summarize_youtube_video 内分支）
summarize_video(ollama, video_url, "llama3.2")


Model used: llama3.2


### Summary

Here is a summary of the transcript:

The president stated that their new military campaign has stopped record amounts of drugs from entering the country, almost completely halted smuggling by water or sea, and successfully taken down a powerful cartel kingpin, Nicholas Maduro, who was overthrown and brought to face American justice. The success of this operation is seen as a colossal victory for US security and opens up a new beginning for Venezuela.

Key points:
• Large parts of Mexico have been controlled by murderous drug cartels.
• The president declared illicit fentinol a weapon of mass destruction.
• A military campaign has stopped record amounts of drugs from entering the country.
• A powerful cartel kingpin, Nicholas Maduro, was taken down and overthrown.
• Maduro's regime was described as having a "heavily protected" military fortress.

People/organizations mentioned:
- President (implied)
- Deli Rodriguez, new president of Venezuela
- Mexican territory
- Russia
- China

Claims that could not be verified from transcript:
- [unclear] definition or existence of the chemical compound "fentinol"

### 再试试 DeepSeek！

同一套 `summarize_video(ollama, ...)`，只把模型别名换成 `"deepseek"`（内部映射到 `deepseek-r1:1.5b`）。


In [170]:
# ========== 可选清理：删除本机较大的 deepseek-r1:7b ==========

# ollama rm：移除已下载模型，腾出磁盘；若本机没有该模型，命令可能报错（属预期）
!ollama rm deepseek-r1:7b


deleted 'deepseek-r1:7b'


⠙ 


In [172]:
# ========== 拉取较小的 DeepSeek R1 蒸馏版 ==========

# 1.5b 参数更小，适合本机试跑；名称须与函数内 m = "deepseek-r1:1.5b" 一致
!ollama pull deepseek-r1:1.5b


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling aabd4debf0c8:   0% ▕                  ▏  16 KB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏ 6.5 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   1% ▕                  ▏  14 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  18 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   2% ▕                  ▏  25 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  32 MB/1.1 GB                  pulling manifest 
pulling aabd4debf0c8:   3% ▕                  ▏  38 MB/1.1

In [179]:
# ========== 试跑 C：同一 video_url，用 deepseek 别名走 Ollama ==========

# model_="deepseek" → 内部选用 deepseek-r1:1.5b；需先跑完上面的 pull 与 ollama 客户端创建格
summarize_video(ollama, video_url, "deepseek")


Model used: deepseek-r1:1.5b


### Summary

(1-2 sentences)  
The video discusses how large territories controlled by drug cartels used fentinol as a weapon to stop illegal drug flow and take down key figures, ultimately defeating Donald Maduro in the US. Key people mentioned include drug cartel members and the use of fentinol.  

People/organizations involved: Drug cartels, military forces. Claims not verifiable due to unknown individuals and context.  
Unclear aspects: [large state terms translated as "territories," other variables possibly implied.]